In [5]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error


df = pd.read_csv('modified_digital_media_with_business_segment.csv')


X = df.drop(columns=['campaign_id','revenue_usd','perfomance_tier_index'], errors='ignore')
y = df['revenue_usd']


from sklearn.preprocessing import OrdinalEncoder

cat_cols = X.select_dtypes(include='object').columns

encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

X[cat_cols] = encoder.fit_transform(X[cat_cols])


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

best_model1 = XGBRegressor(
    subsample=0.6,
    reg_lambda=2,
    reg_alpha=1,
    n_estimators=1500,
    min_child_weight=3,
    max_depth=2,
    learning_rate=0.01,
    gamma=0.5,
    colsample_bytree=1.0,
    colsample_bynode=0.8,
    colsample_bylevel=0.8,

    random_state=42,
    objective='reg:squarederror',
    enable_categorical=True
)


kf = KFold(n_splits=5, shuffle=True, random_state=42)

cv_scores = cross_val_score(
    best_model1,
    X_train,
    y_train,
    cv=kf,
    scoring='r2',
    n_jobs=-1
)

val_r2 = cv_scores.mean()


best_model1.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

y_train_pred = best_model1.predict(X_train)
y_test_pred = best_model1.predict(X_test)


train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print("\n" + "="*60)
print(f"{'Metric':<25} | {'R2 Score':<10} | {'RMSE':<10}")
print("-" * 60)

print(f"{'Training':<25} | {train_r2:.4f}     | {train_rmse:.2f}")
print(f"{'Validation (5-Fold CV)':<25} | {val_r2:.4f}     | N/A")
print(f"{'Test':<25} | {test_r2:.4f}     | {test_rmse:.2f}")

print("="*60)


print("\nCV Scores:", cv_scores)
print(f"CV Mean: {cv_scores.mean():.4f}")
print(f"CV Std Dev: {cv_scores.std():.4f}")


Metric                    | R2 Score   | RMSE      
------------------------------------------------------------
Training                  | 0.9214     | 2862.38
Validation (5-Fold CV)    | 0.8808     | N/A
Test                      | 0.8834     | 3544.66

CV Scores: [0.83851463 0.89840268 0.87741354 0.90055769 0.88913321]
CV Mean: 0.8808
CV Std Dev: 0.0227


In [2]:
import joblib

joblib.dump(best_model1, "xgb_reg.pkl")
joblib.dump(encoder, "encoder.pkl")  # overwrite OK if same dataset
joblib.dump(X.columns.tolist(), "features.pkl")
joblib.dump(cat_cols.tolist(), "cat_cols.pkl")

['cat_cols.pkl']